In [1]:
from pathlib import Path
from typing import Dict, List, Tuple
from docplex.mp.dvar import Var
from docplex.mp.model import Model


def ler_instancia_pt(
    caminho: Path,
) -> Tuple[int, int, List[int], List[int], List[List[float]]]:
    linhas = [
        linha.strip()
        for linha in caminho.read_text(encoding="utf-8").splitlines()
        if linha.strip()
    ]
    m, n = map(int, linhas[0].split())
    ofertas = [int(v) for v in linhas[1].split()]
    demandas = [int(v) for v in linhas[2].split()]
    custos = [[float(v) for v in linha.split()] for linha in linhas[3 : 3 + m]]
    return m, n, ofertas, demandas, custos


m, n, ofertas, demandas, custos = ler_instancia_pt(Path("in.txt"))
print(f"Instância carregada: {m} origens e {n} destinos.")

Instância carregada: 3 origens e 4 destinos.


In [2]:
mdl = Model(name="Problema_do_Transporte")

# Variáveis inteiras não-negativas x_ij (unidades transportadas da origem i ao destino j)
x: Dict[Tuple[int, int], Var] = {
    (i, j): mdl.integer_var(lb=0, name=f"x_{i + 1}_{j + 1}")
    for i in range(m)
    for j in range(n)
}

# (1) Função Objetivo: minimizar o custo total de transporte
mdl.minimize(
    mdl.sum(custos[i][j] * x[(i, j)] for i in range(m) for j in range(n))
)

# (2) Atender completamente a demanda de cada destino j
for j in range(n):
    mdl.add_constraint(
        mdl.sum(x[(i, j)] for i in range(m)) == demandas[j],
        ctname=f"demanda_{j + 1}",
    )

# (3) Respeitar a oferta disponível de cada origem i
for i in range(m):
    mdl.add_constraint(
        mdl.sum(x[(i, j)] for j in range(n)) <= ofertas[i],
        ctname=f"oferta_{i + 1}",
    )

print("Modelo PT construído com sucesso.")

Modelo PT construído com sucesso.


In [3]:
sol = mdl.solve(log_output=False)

if sol is None:
    print("Nenhuma solução encontrada.")
else:
    print(f"Status: {mdl.solve_details.status}")
    print(f"Função Objetivo (Custo Mínimo): {sol.objective_value:.2f}\n")
    print("Unidades transportadas (Origem -> Destino):")
    for i in range(m):
        for j in range(n):
            val = int(round(sol.get_value(x[(i, j)])))
            print(f"  x[{i + 1}->{j + 1}] = {val:>2d}   (custo unit.: {custos[i][j]:.0f})")

Status: integer optimal solution
Função Objetivo (Custo Mínimo): 1330.00

Unidades transportadas (Origem -> Destino):
  x[1->1] =  5   (custo unit.: 14)
  x[1->2] =  0   (custo unit.: 16)
  x[1->3] = 25   (custo unit.: 13)
  x[1->4] =  0   (custo unit.: 18)
  x[2->1] = 15   (custo unit.: 8)
  x[2->2] =  1   (custo unit.: 9)
  x[2->3] =  0   (custo unit.: 10)
  x[2->4] = 34   (custo unit.: 11)
  x[3->1] =  0   (custo unit.: 18)
  x[3->2] = 27   (custo unit.: 16)
  x[3->3] =  0   (custo unit.: 21)
  x[3->4] =  0   (custo unit.: 20)
